# core — one wrap, and every tool downstream can see the call

`instrument()` is the whole integration. It identifies an LLM client by its **shape**, wraps it once, and from then on every call lands on a normalized bus with exact usage and a `Decimal` cost. Nothing else in your code changes.

> **Offline.** No API key, no network — the provider is a fake with the real client's *shape*, or a
> committed cassette. This notebook runs in CI on Python 3.11 and 3.13 via `nbmake`, so if a cell
> below stops working the build goes red.
>
> Beside it, [`main.py`](main.py) is the same story as a script. The last cell here asserts what
> that script asserts.

In [ ]:
# The notebook sits beside the recipe, so its own module is importable. Everything below reuses the
# recipe's fixtures rather than re-inventing them — a notebook that built its own fake could drift
# away from what `main.py` proves and nobody would notice.
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd()))

## 1 · A client with the right shape

This is what makes the whole cookbook keyless: `instrument()` never checks the class name or reaches for the network, so a `SimpleNamespace` with a `chat.completions.create` method is recognised exactly like a real `OpenAI()`.

In [ ]:
import main as recipe
from cendor.core import bus, instrument

fake = recipe.fake_openai()
type(fake), type(fake.chat.completions)

## 2 · Subscribe, then wrap

Any tool subscribes the same way — this is the seam `tokenguard`, `acttrace` and `cassette` all sit on.

In [ ]:
seen = []
bus.subscribe(seen.append)
client = instrument(fake)  # the one and only wrap
client

## 3 · Make a call

Ordinary provider code. Nothing here is cendor-aware.

In [ ]:
client.chat.completions.create(model="gpt-4o", messages=[{"role": "user", "content": "hello"}])
call = seen[-1]
call.provider, call.model

## 4 · What the bus received

Usage is normalized out of the provider's own shape; the cost comes from `prices`, not from anything typed into the recipe.

In [ ]:
from cendor.core import tokens

print(f"usage : {call.usage.input_tokens} in / {call.usage.output_tokens} out")
print(f"cost  : ${call.cost.amount}")
print(f"tokens: counted via {tokens.method(call.model)!r}")

## 5 · Prove it

A seam that silently emitted nothing would print nothing and still exit 0, which is why `main.py` ends in these four lines rather than a paragraph.

In [ ]:
assert seen, "no event reached the bus — instrument() captured nothing"
assert call.provider == "openai"
assert call.usage.input_tokens and call.usage.output_tokens
assert call.cost and call.cost.amount > 0
print("OK")